# NISS ablation - AZ v4 V + sym4, strat-51, production multi-pass

Does `--niss` earn its 2x wall in the production full-1001 recipe? Run the
production solver (`03_solve.py`) on the 51 stratified pids (incl. hard pids
901/907/911/970/992/1000) at `--sym-ensemble 4 --beams 16384,65536
--max-steps 60,150 --bf16`, once **without** NISS and once **with**. Compare
solve count + total moves (model-or-fallback) over the 51 pids.


In [ ]:

import os, sys, glob, subprocess, time
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

hits = glob.glob("/kaggle/input/**/megaminx/scripts/03_solve.py", recursive=True)
assert hits, f"03_solve.py not found; {sorted(glob.glob('/kaggle/input/**', recursive=True))[:30]}"
BUNDLE = hits[0].split("/megaminx/scripts/")[0]
print("mount:", BUNDLE, flush=True)

import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available(),
      (torch.cuda.get_device_name(0) if torch.cuda.is_available() else ""), flush=True)

script = f"{BUNDLE}/megaminx/scripts/03_solve.py"
ckpt = f"{BUNDLE}/m_az_v4_v_only.pt"
common = [sys.executable, script, "--checkpoint", ckpt,
          "--stratified", "5", "--strat-seed", "0",
          "--sym-ensemble", "4",
          "--beams", "16384,65536", "--max-steps", "60,150", "--bf16"]

def run(extra, out, tag):
    cmd = common + extra + ["--out", out]
    print(f"\n===== ARM {tag}: {' '.join(cmd)} =====", flush=True)
    t0 = time.time()
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait()
    print(f"ARM {tag} exit {p.returncode}  wall {time.time() - t0:.0f}s", flush=True)

run([], "/kaggle/working/strat51_sym4_noniss.csv", "A_noniss")
run(["--niss"], "/kaggle/working/strat51_sym4_niss.csv", "B_niss")


In [ ]:

import random, csv
all_ids = list(range(1001)); rng = random.Random(0); buckets = {}
for pid in all_ids:
    buckets.setdefault(pid // 100, []).append(pid)
strat = []
for b in sorted(buckets):
    strat.extend(sorted(rng.sample(buckets[b], min(5, len(buckets[b])))))
strat = set(strat)
print(f"strat-51 pids: {len(strat)}")

def load(p):
    d = {}
    with open(p) as f:
        for row in csv.DictReader(f):
            pid = int(row["initial_state_id"]); path = row["path"]
            d[pid] = 0 if not path else len(path.split("."))
    return d

A = load("/kaggle/working/strat51_sym4_noniss.csv")
B = load("/kaggle/working/strat51_sym4_niss.csv")
TH = 200  # model paths ~50-160; pp_bfs6 fallback ~400 -> clean split

def summ(d, name):
    solved = sorted(p for p in strat if d[p] < TH)
    total = sum(d[p] for p in strat)
    mtotal = sum(d[p] for p in solved)
    print(f"{name}: model-solved {len(solved)}/51 | strat-51 total(model-or-fb) {total} | model-only total {mtotal}")
    return set(solved), total

print()
sA, tA = summ(A, "A no-niss")
sB, tB = summ(B, "B niss   ")
print(f"\nNISS delta on strat-51 total: {tB - tA:+d}  (negative = niss better)")
print(f"solved only WITH niss:    {sorted(sB - sA)}")
print(f"solved only WITHOUT niss: {sorted(sA - sB)}")
common = sA & sB
shaved = sum(A[p] - B[p] for p in common)
print(f"on {len(common)} commonly-solved pids, niss shaved {shaved:+d} moves total "
      f"(positive = niss shorter)")
worse = [p for p in common if B[p] > A[p]]
print(f"  pids where niss was LONGER: {sorted(worse)}")
